## Imports

In [ ]:
import json, pickle, random, math
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from tokenizers import Tokenizer

## Loading the existing preprocessed data

In [5]:
working_dir = Path.cwd().resolve()
PROJECT_DIR = working_dir.parent if working_dir.name == "notebooks" else working_dir
PROCESSED_DIR = PROJECT_DIR/"data"/"processed"
MODEL_DIR = PROJECT_DIR/"models"/"attention"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for filename in ["tokenizer.json", "encoded_lines.pkl", "vocab.json"]:
    if not (PROCESSED_DIR/filename).exists():
        raise FileNotFoundError(f"Missing {PROCESSED_DIR/filename}. Run preprocessing first.")

print("Project:", PROJECT_DIR)
print("Checkpoints:", MODEL_DIR)

Project: /home/gusvahaye@GU.GU.SE/Smart-Autocomplete-Systems
Checkpoints: /home/gusvahaye@GU.GU.SE/Smart-Autocomplete-Systems/models/attention


In [6]:
hf_tokenizer = Tokenizer.from_file(str(PROCESSED_DIR/"tokenizer.json"))

with open(PROCESSED_DIR/"encoded_lines.pkl", "rb") as f:
    encoded_lines = pickle.load(f)

with open(PROCESSED_DIR/"vocab.json", "r", encoding="utf-8") as f:
    vocab_data = json.load(f)

word_to_id = vocab_data["word_to_id"]
id_to_word = {int(idx): word for word, idx in word_to_id.items()}

vocab_size = len(word_to_id)

bos_id = word_to_id["<bos>"]
eos_id = word_to_id["<eos>"]
unk_id = word_to_id["<unk>"]
pad_id = word_to_id["<pad>"]

---

## settings

In [7]:
SEED = 42
def set_seed(seed=SEED):
    random.seed(seed) # python rng
    np.random.seed(seed) # numpy rng
    torch.manual_seed(seed) # pytorch rng
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BLOCK_SIZE = 32
BATCH_SIZE = 4
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 2
FF_DIM = 256
DROPOUT = 0.2
LEARNING_RATE = 0.001
EPOCHS = 10

In [8]:
device

device(type='cuda')

### creating training blocks

mostly like what we did in lstm

In [11]:
def make_transformer_dataset(sequences, block_size, pad_id):
    inputs = []
    targets = []
    
    for sequence in sequences:
        for start in range(0, len(sequence) - 1, block_size):
            # just like lstm
            end = min(start + block_size, len(sequence) - 1)
            
            x = list(sequence[start:end])
            y = list(sequence[start + 1:end + 1])
            
            inputs.append(x+[pad_id]*(block_size - len(x)))
            targets.append(y+[pad_id]*(block_size - len(y)))
            
    input_blocks = torch.tensor(inputs, dtype=torch.long)
    output_blocks = torch.tensor(targets, dtype=torch.long)
            
    return TensorDataset(input_blocks,output_blocks)

train_dataset = make_transformer_dataset(encoded_lines["train"], BLOCK_SIZE, pad_id)
validation_dataset = make_transformer_dataset(encoded_lines["validation"], BLOCK_SIZE, pad_id)
test_dataset = make_transformer_dataset(encoded_lines["test"], BLOCK_SIZE, pad_id)

print("Training blocks:", len(train_dataset))
print("Validation blocks:", len(validation_dataset))
print("Test blocks:", len(test_dataset))

def make_train_loader(seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)

validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

x_example, y_example = train_dataset[0]

print("\nInput ids:", x_example[:8].tolist())
print("Target ids:", y_example[:8].tolist())

Training blocks: 70766
Validation blocks: 7386
Test blocks: 8390

Input ids: [2, 21646, 119, 3840, 152, 44, 1, 4345]
Target ids: [21646, 119, 3840, 152, 44, 1, 4345, 23]


---

## Decoder-only transformer with causal self-attention

In [12]:
class AttentionLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, n_layers=2, ff_dim=256, dropout=0.2, block_size=32, pad_id=0):
        super().__init__()
        self.block_size = block_size
        self.pad_id = pad_id
        
        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(block_size, d_model)
        
        # transformer layer
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim, dropout=dropout, batch_first=True, norm_first=True, activation="gelu")
        
        self.transformer = nn.TransformerEncoder(layer, num_layers=n_layers, enable_nested_tensor=False) # stacking transformer layers
        self.final_norm = nn.LayerNorm(d_model) # normalization
        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids):
        batch, length = input_ids.shape
    
        positions = torch.arange(length, device=input_ids.device).unsqueeze(0)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        
        # True means attention is blocked so future positions are invisible
        causal_mask = torch.triu(torch.ones(length, length, dtype=torch.bool, device=input_ids.device), diagonal=1)

        x = self.transformer(x, mask=causal_mask) # pass it to transformer
        return self.output(self.final_norm(x))

In [15]:
MODEL_CONFIG = {
    "vocab_size": vocab_size,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "ff_dim": FF_DIM,
    "dropout": DROPOUT,
    "block_size": BLOCK_SIZE,
    "pad_id": pad_id
}

model = AttentionLanguageModel(**MODEL_CONFIG).to(device)

total_params = sum(p.numel() for p in model.parameters()) # counting parameters
print("Parameters:", f"{total_params:,}")

# then to test first we have to prepare the example input
example_input = x_example.unsqueeze(0) # add one dimensioin at index 0
example_input = example_input.to(device)

print("Input shape:", example_input.shape)

# then testing the model
model.eval()
with torch.no_grad():
    output = model(example_input)

print("Output shape:", output.shape)

## checking the dimension
expected_shape = (1, BLOCK_SIZE, vocab_size)
assert output.shape == expected_shape

print("Forward-pass shape test passed")

Parameters: 12,133,717
Input shape: torch.Size([1, 32])
Output shape: torch.Size([1, 32, 46165])
Forward-pass shape test passed


---

## Training and evaluation for one full epoch

In [16]:
# Loss function
criterion = nn.CrossEntropyLoss(ignore_index=pad_id, reduction="sum")

def run_epoch(model, data_loader, criterion, optimizer=None):
    # like the way i wrote the same function for lstm, if we provide optimizer the function will be training, otherwise evaluation
    training = optimizer is not None
    model.train(training)

    total_loss = 0
    total_tokens = 0

    for inputs, targets in data_loader:

        inputs = inputs.to(device)
        targets = targets.to(device)

        valid_tokens = (targets != pad_id).sum().item() # Count tokens that are not padding
        if valid_tokens == 0:
            continue

        # making prediction
        with torch.set_grad_enabled(training):
            logits = model(inputs)

            # loss
            loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1))

            if training:
                optimizer.zero_grad()

                # gradients
                (loss / valid_tokens).backward()

                # to prevent very large gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                optimizer.step()

        # Save loss and token count
        total_loss += loss.item()
        total_tokens += valid_tokens

    if total_tokens == 0:
        raise ValueError("No valid tokens in DataLoader")

    average_loss = total_loss / total_tokens
    perplexity = math.exp(min(average_loss, 20))

    return {
        "loss": average_loss,
        "perplexity": perplexity,
        "tokens": total_tokens
    }


print("Training batches:", len(make_train_loader()))
print("Validation batches:", len(validation_loader))

Training batches: 17692
Validation batches: 1847
